> **BRIEF (WS2, owner: Carlos)** (not copied into the final notebook)
> Details in `docs/WORK_PLAN.md`.
>
> 1. Flip one Wonderland input at a time; full 8-row table x → z → σ(z) → decision
> 2. Explain why raw z and σ(z) are not interchangeable when computing error
> 3. Prostate Cancer ANN from the *Deep Learning Fundamentals* class notes: Markdown + code + **3 talking points**
>    The class notes are not in the repo. Get the slide first.
> Restart & Run All before pushing.

In [1]:
import torch
from itertools import product

### Wonderland: changing one input at a time

Canada's Wonderland neuron has three yes/no inputs, weights $w = [5, 2, 4]$ and bias $b = -3$. It computes the raw score $z = w \cdot x + b$ and then the prediction $\sigma(z) = \dfrac{1}{1 + e^{-z}}$. We decide "yes" when $\sigma(z) \geq 0.5$ (which happens exactly when $z \geq 0$).

We start from $x = [1, 0, 1]$ and flip **one input at a time**.

In [2]:
w = torch.tensor([5., 2., 4.])
b = torch.tensor(-3.)

def neuron(x):
    z = w @ x + b            # raw score
    a = torch.sigmoid(z)     # activated prediction
    return z, a

x0 = torch.tensor([1., 0., 1.])
z0, a0 = neuron(x0)
print(f"start  x = {x0.int().tolist()}   z = {z0.item():5.1f}   sigmoid(z) = {a0.item():.4f}   decision = {int(a0 >= 0.5)}")
print("-" * 70)

for i in range(3):
    x = x0.clone()
    x[i] = 1 - x[i]          # flip input i only
    z, a = neuron(x)
    print(f"flip x{i+1} x = {x.int().tolist()}   z = {z.item():5.1f}   sigmoid(z) = {a.item():.4f}   decision = {int(a >= 0.5)}   (change in z: {z.item() - z0.item():+.0f})")

start  x = [1, 0, 1]   z =   6.0   sigmoid(z) = 0.9975   decision = 1
----------------------------------------------------------------------
flip x1 x = [0, 0, 1]   z =   1.0   sigmoid(z) = 0.7311   decision = 1   (change in z: -5)
flip x2 x = [1, 1, 1]   z =   8.0   sigmoid(z) = 0.9997   decision = 1   (change in z: +2)
flip x3 x = [1, 0, 0]   z =   2.0   sigmoid(z) = 0.8808   decision = 1   (change in z: -4)


Each flip moves $z$ by exactly the weight of that input, up or down: turning $x_1$ off lowers $z$ by 5, turning $x_2$ on raises it by 2, and turning $x_3$ off lowers it by 4. None of the three single flips changes the decision, because the starting score $z = 6$ is far above 0.

Now the full table of all $2^3 = 8$ input combinations.

In [3]:
print(" x1 x2 x3 |    z  | sigmoid(z) | decision")
print("-" * 40)
for bits in product([0., 1.], repeat=3):
    x = torch.tensor(bits)
    z, a = neuron(x)
    print(f"  {int(bits[0])}  {int(bits[1])}  {int(bits[2])} | {z.item():4.0f}  |   {a.item():.4f}   |    {int(a >= 0.5)}")

 x1 x2 x3 |    z  | sigmoid(z) | decision
----------------------------------------
  0  0  0 |   -3  |   0.0474   |    0
  0  0  1 |    1  |   0.7311   |    1
  0  1  0 |   -1  |   0.2689   |    0
  0  1  1 |    3  |   0.9526   |    1
  1  0  0 |    2  |   0.8808   |    1
  1  0  1 |    6  |   0.9975   |    1
  1  1  0 |    4  |   0.9820   |    1
  1  1  1 |    8  |   0.9997   |    1


The neuron says "no" only for $[0,0,0]$ and $[0,1,0]$, the two rows where both $x_1$ and $x_3$ are off. Input $x_2$ has a small weight (2), so it can never rescue a row on its own. The inputs $x_1$ and $x_3$ carry the decision.

### Raw score $z$ vs prediction $\sigma(z)$

For the starting point $x = [1, 0, 1]$ we have $z = 6$ and $\sigma(6) \approx 0.9975$. They look related, but **the raw score $z = 6$ and the activated prediction $\sigma(6) \approx 0.9975$ are not interchangeable when computing classification error.**

The reason is the scale. The target is a label that is either 0 or 1, and only the activated prediction lives on that scale. The raw score $z$ can be any real number, so comparing it to a 0/1 target gives a meaningless error. The cell below shows this for a target of $y = 1$ and for a wrong target of $y = 0$.

In [4]:
z = torch.tensor(6.)
a = torch.sigmoid(z)
print(f"z = {z.item():.1f}   sigmoid(z) = {a.item():.4f}\n")

for y in (1., 0.):
    err_raw = y - z.item()      # wrong: compares the raw score with a 0/1 label
    err_act = y - a.item()      # right: compares the prediction with a 0/1 label
    print(f"target y = {y:.0f}:  error with z = {err_raw:+.4f} (squared loss {0.5 * err_raw**2:8.4f})"
          f"   error with sigmoid(z) = {err_act:+.4f} (squared loss {0.5 * err_act**2:.6f})")

# the slope of the sigmoid is also tiny at z = 6, so learning is slow where the neuron is already confident
print(f"\nsigmoid'(6) = sigmoid(6) * (1 - sigmoid(6)) = {(a * (1 - a)).item():.5f}")

z = 6.0   sigmoid(z) = 0.9975

target y = 1:  error with z = -5.0000 (squared loss  12.5000)   error with sigmoid(z) = +0.0025 (squared loss 0.000003)
target y = 0:  error with z = -6.0000 (squared loss  18.0000)   error with sigmoid(z) = -0.9975 (squared loss 0.497530)

sigmoid'(6) = sigmoid(6) * (1 - sigmoid(6)) = 0.00247


With $y = 1$, the raw score gives an error of $-5$ even though the neuron is already almost perfectly right. With $y = 0$, it gives $-6$, while the real error is just below $1$ because $\sigma(z)$ can never exceed 1. Only $\sigma(z)$ gives an error bounded between 0 and 1 that matches the 0/1 label, so the loss is computed on $\sigma(z)$. The raw $z$ is still needed as the input to the sigmoid and for the decision rule $z \geq 0$.

### Prostate Cancer Prediction ANN

**Problem.** Given a patient's (scaled) PSA value $x$, predict whether the patient has prostate cancer or is healthy. The labels are not monotonic in PSA: Cancer appears at high values and at low values, while Healthy sits in the middle (about 1.4 to 2.1). A single neuron can only cut the PSA axis once, so the network needs **two hidden neurons** (two cut points).

**Network.** 1 input $\rightarrow$ 2 hidden neurons $\rightarrow$ 2 output neurons (Cancer, Healthy). All neurons use the sigmoid, as in the Wonderland case. Targets are encoded as Cancer $= [1, 0]$ and Healthy $= [0, 1]$.

Parameters taken from the class notes:

| Neuron | Weight(s) | Bias |
|---|---|---|
| Hidden 1 | $w_{11} = 26.19$ | $b_1 = -34.59$ |
| Hidden 2 | $w_{21} = 3.38$ | $b_2 = -8.25$ |
| Output Cancer | $w_{c,1} = -64.51$, $w_{c,2} = 88.09$ | $b_c = 23.04$ |
| Output Healthy | $w_{h,1} = 63.97$, $w_{h,2} = -87.57$ | $b_h = -22.59$ |

**Step 1: forward pass on all 15 patients.**

In [5]:
# 15 patients: PSA value (x) and status
status = ["Cancer"] * 5 + ["Healthy"] * 5 + ["Cancer"] * 5
x = torch.tensor([3.8, 3.4, 2.9, 2.8, 2.7,
                  2.1, 2.0, 1.7, 1.6, 1.4,
                  0.9, 0.8, 0.2, 0.2, 0.1]).reshape(-1, 1)          # shape (15, 1)
Y = torch.tensor([[1., 0.] if s == "Cancer" else [0., 1.] for s in status])   # shape (15, 2)

# parameters from the class notes (one row per neuron)
W1 = torch.tensor([[26.19], [3.38]]);              b1 = torch.tensor([-34.59, -8.25])
W2 = torch.tensor([[-64.51, 88.09], [63.97, -87.57]]); b2 = torch.tensor([23.04, -22.59])   # rows: Cancer, Healthy

def forward(x, W1, b1, W2, b2):
    h = torch.sigmoid(x @ W1.T + b1)       # hidden activations, shape (n, 2)
    y_hat = torch.sigmoid(h @ W2.T + b2)   # outputs [Cancer, Healthy], shape (n, 2)
    return h, y_hat

h, y_hat = forward(x, W1, b1, W2, b2)

print("  PSA  status  | hidden (h1, h2)  | output (Cancer, Healthy) | predicted")
print("-" * 76)
correct = 0
for i in range(len(x)):
    pred = "Cancer" if y_hat[i, 0] > y_hat[i, 1] else "Healthy"
    correct += pred == status[i]
    print(f"  {x[i].item():.1f}  {status[i]:<7} | ({h[i,0]:.3f}, {h[i,1]:.3f})   |     ({y_hat[i,0]:.3f}, {y_hat[i,1]:.3f})        | {pred}")
print(f"\ncorrect: {correct} / {len(x)}")

  PSA  status  | hidden (h1, h2)  | output (Cancer, Healthy) | predicted
----------------------------------------------------------------------------
  3.8  Cancer  | (1.000, 0.990)   |     (1.000, 0.000)        | Cancer
  3.4  Cancer  | (1.000, 0.962)   |     (1.000, 0.000)        | Cancer
  2.9  Cancer  | (1.000, 0.825)   |     (1.000, 0.000)        | Cancer
  2.8  Cancer  | (1.000, 0.771)   |     (1.000, 0.000)        | Cancer
  2.7  Cancer  | (1.000, 0.706)   |     (1.000, 0.000)        | Cancer
  2.1  Healthy | (1.000, 0.240)   |     (0.000, 1.000)        | Healthy
  2.0  Healthy | (1.000, 0.184)   |     (0.000, 1.000)        | Healthy
  1.7  Healthy | (1.000, 0.076)   |     (0.000, 1.000)        | Healthy
  1.6  Healthy | (0.999, 0.055)   |     (0.000, 1.000)        | Healthy
  1.4  Healthy | (0.889, 0.029)   |     (0.000, 1.000)        | Healthy
  0.9  Cancer  | (0.000, 0.005)   |     (1.000, 0.000)        | Cancer
  0.8  Cancer  | (0.000, 0.004)   |     (1.000, 0.000)        | 

The class-notes parameters classify all 15 patients correctly, and the outputs are already almost exactly 0 or 1. So this is a **trained** network. We now follow the Wonderland pattern for **one gradient-descent step**: forward pass, error, gradient, update.

**Step 2: error and gradients.** With the loss $L = \tfrac{1}{2}\sum_k (y_k - \hat{y}_k)^2$ and sigmoid outputs, the gradient of an output weight is $\dfrac{\partial L}{\partial w} = (\hat{y} - y)\,\hat{y}(1 - \hat{y})\,\cdot$ (input of that weight), the same form as in the Wonderland case. The error is then passed back through the hidden layer with the chain rule. The function below computes every gradient by hand and checks it against PyTorch's autograd.

**Step 3: update** each parameter with $w \leftarrow w - \eta\,\dfrac{\partial L}{\partial w}$, using $\eta = 0.1$ (our assumption, the class notes do not give a learning rate).

In [6]:
eta = 0.1   # learning rate: our assumption

def one_step(psa, target):
    xi = torch.tensor([[psa]])
    yi = torch.tensor([target])

    # forward pass, with autograd tracking
    params = [t.clone().requires_grad_() for t in (W1, b1, W2, b2)]
    h_i, out_i = forward(xi, *params)
    loss = 0.5 * ((yi - out_i) ** 2).sum()
    loss.backward()

    # the same gradients by hand (chain rule)
    d_out = (out_i - yi) * out_i * (1 - out_i)       # dL/dz at the output neurons
    d_hid = (d_out @ W2) * h_i * (1 - h_i)           # dL/dz at the hidden neurons
    manual = [d_hid.T @ xi, d_hid.squeeze(0), d_out.T @ h_i, d_out.squeeze(0)]
    same = all(torch.allclose(m, p.grad, atol=1e-6) for m, p in zip(manual, params))

    # one gradient-descent step
    with torch.no_grad():
        new = [p - eta * p.grad for p in params]
    _, out_new = forward(xi, *new)
    loss_new = 0.5 * ((yi - out_new) ** 2).sum()

    print(f"PSA = {psa}   target = {target}")
    print(f"  output (Cancer, Healthy) = ({out_i[0,0]:.4f}, {out_i[0,1]:.4f})   error = ({(yi - out_i)[0,0]:+.4f}, {(yi - out_i)[0,1]:+.4f})   loss = {loss.item():.4e}")
    print(f"  hand gradients equal autograd: {same}")
    for n, p, nw in zip(["w_11, w_21        ", "b_1, b_2          ", "w_c1, w_c2, w_h1, w_h2", "b_c, b_h          "], params, new):
        print(f"  {n}  grad = {[float(f'{v:.3e}') for v in p.grad.flatten().tolist()]}")
        print(f"  {'':<22}  new  = {[round(v, 4) for v in nw.flatten().tolist()]}")
    print(f"  loss after the step = {loss_new.item():.4e}\n")

# (a) a real patient from the class data: the one with the largest loss (PSA 2.7, Cancer)
one_step(2.7, [1., 0.])

PSA = 2.7   target = [1.0, 0.0]
  output (Cancer, Healthy) = (1.0000, 0.0000)   error = (+0.0000, -0.0000)   loss = 8.7444e-19
  hand gradients equal autograd: True
  w_11, w_21          grad = [0.0, -8.583e-17]
                          new  = [26.19, 3.38]
  b_1, b_2            grad = [0.0, -3.179e-17]
                          new  = [-34.59, -8.25]
  w_c1, w_c2, w_h1, w_h2  grad = [0.0, 0.0, 1.749e-18, 1.235e-18]
                          new  = [-64.51, 88.09, 63.97, -87.57]
  b_c, b_h            grad = [0.0, 1.749e-18]
                          new  = [23.04, -22.59]
  loss after the step = 8.7444e-19



For a real patient the loss is around $10^{-18}$ and every gradient is essentially zero, so the update changes nothing. The network is **saturated**: the weights are large, so $z$ is far from 0, $\hat{y}$ sits at 0 or 1, and the sigmoid slope $\hat{y}(1 - \hat{y})$ is almost 0. This is the same lesson as the raw score $z$ vs $\sigma(z)$ in Wonderland.

To see a visible step we need a patient the network is unsure about. The next cell uses an **illustrative patient** that is *not* in the class data: PSA $= 1.3$, which sits right next to the cut point of hidden neuron 1, labelled Healthy.

In [7]:
# (b) illustrative borderline patient (NOT in the class data): PSA = 1.3, Healthy
one_step(1.3, [0., 1.])

PSA = 1.3   target = [0.0, 1.0]
  output (Cancer, Healthy) = (0.7610, 0.2900)   error = (-0.7610, +0.7100)   loss = 5.4161e-01
  hand gradients equal autograd: True
  w_11, w_21          grad = [-5.524, 0.6591]
                          new  = [26.7424, 3.3141]
  b_1, b_2            grad = [-4.249, 0.507]
                          new  = [-34.1651, -8.3007]
  w_c1, w_c2, w_h1, w_h2  grad = [0.05087, 0.002867, -0.05372, -0.003028]
                          new  = [-64.5151, 88.0897, 63.9754, -87.5697]
  b_c, b_h            grad = [0.1384, -0.1462]
                          new  = [23.0262, -22.5754]
  loss after the step = 8.0266e-16



### Talking Points

1. **Two hidden neurons give two cut points on the PSA axis.** Each hidden neuron switches at the PSA where its score is 0: hidden 1 at $34.59 / 26.19 \approx 1.32$ and hidden 2 at $8.25 / 3.38 \approx 2.44$. The Healthy patients (1.4 to 2.1) are exactly the ones between the two cuts, with hidden 1 on and hidden 2 off. The Healthy output neuron uses a positive weight on hidden 1 and a negative weight on hidden 2 to detect this band, and the Cancer neuron fires for everything outside it. One neuron could not do this, because Cancer appears on both sides of Healthy.
2. **The trained network is saturated, so a gradient step barely moves it.** The weights are large (tens), so $z$ is far from 0, $\sigma(z)$ is almost exactly 0 or 1, and the slope $\hat{y}(1-\hat{y})$ is almost 0. For every real patient the loss is tiny (about $10^{-18}$ for the worst one, PSA 2.7) and the update changes nothing. For a borderline patient (PSA 1.3, illustrative, not in the class data) the loss is about 0.54 and the gradients are large, for example $\partial L/\partial w_{11} \approx -5.5$. This is the Wonderland lesson again: the activated prediction, not the raw score, decides the error, and the sigmoid slope controls how much we learn.
3. **The two outputs are independent sigmoids, not a softmax.** The Cancer and Healthy outputs do not have to add up to 1 (at PSA 1.3 they are 0.76 and 0.29), so we predict by comparing them and the loss sums the error over both outputs. The size of one step is $\eta \times$ gradient, and the learning rate $\eta = 0.1$ is our assumption. With these large weights, even this small $\eta$ is enough: one step on the PSA 1.3 patient takes $w_{11}$ from 26.19 to about 26.74 and the loss from 0.54 to nearly 0. We also checked that our hand-written gradients match PyTorch's autograd.
